# Experiment 6: LightGBM detailed hyperparameter tuning

100 Optuna trials over 9 LightGBM hyperparameters; every trial is logged to MLflow.

Fixes compared with the original notebook:
- **Data leakage.** The original fitted TF-IDF and SMOTE on the *whole* dataset before splitting,
  so synthetic copies of test comments ended up in training and inflated the score.
  Here the data is split first; TF-IDF and SMOTE only ever see training rows.
- **Tuning on the test set.** Trials are scored on a validation split; the test set is used once at the end.
- **`subsample` did nothing.** LightGBM ignores `subsample` unless `subsample_freq > 0`; it is set to 1.
- Only the best model's artifact is logged (logging 100 models costs gigabytes for no benefit).

In [ ]:
import os
from datetime import datetime

import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import optuna
import pandas as pd
import seaborn as sns
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

mlflow.set_tracking_uri(os.getenv('MLFLOW_TRACKING_URI', 'http://127.0.0.1:5000'))
EXPERIMENT = 'Exp 6 - LightGBM HP Tuning'
mlflow.set_experiment(EXPERIMENT)
BATCH = datetime.now().strftime('%Y%m%d-%H%M%S')
optuna.logging.set_verbosity(optuna.logging.WARNING)

N_TRIALS = 100
NGRAM_RANGE = (1, 3)
MAX_FEATURES = 1000

In [ ]:
df = pd.read_csv('reddit_preprocessing.csv').dropna(subset=['clean_comment'])

# Same label mapping as experiment 5 so results are comparable; reports map 2 back to -1
TO_MODEL = {-1: 2, 0: 0, 1: 1}
TO_LABEL = {v: k for k, v in TO_MODEL.items()}
df['category'] = df['category'].map(TO_MODEL)

X_train_text, X_test_text, y_train_raw, y_test = train_test_split(
    df['clean_comment'], df['category'], test_size=0.2, random_state=42, stratify=df['category']
)
X_fit_text, X_val_text, y_fit_raw, y_val = train_test_split(
    X_train_text, y_train_raw, test_size=0.2, random_state=42, stratify=y_train_raw
)


def build_features(train_text, train_labels, *eval_texts):
    """Fit TF-IDF on the training text, oversample it with SMOTE, and transform the evaluation text."""
    vectorizer = TfidfVectorizer(ngram_range=NGRAM_RANGE, max_features=MAX_FEATURES)
    X = vectorizer.fit_transform(train_text)
    X, y = SMOTE(random_state=42).fit_resample(X, train_labels)
    return (X, y, *[vectorizer.transform(t) for t in eval_texts])


X_fit, y_fit, X_val = build_features(X_fit_text, y_fit_raw, X_val_text)
X_train, y_train, X_test = build_features(X_train_text, y_train_raw, X_test_text)
X_fit.shape, X_val.shape, X_train.shape, X_test.shape

In [ ]:
def build_lgbm(trial):
    return LGBMClassifier(
        n_estimators=trial.suggest_int('n_estimators', 100, 1000),
        learning_rate=trial.suggest_float('learning_rate', 1e-4, 1e-1, log=True),
        max_depth=trial.suggest_int('max_depth', 3, 15),
        num_leaves=trial.suggest_int('num_leaves', 20, 150),
        min_child_samples=trial.suggest_int('min_child_samples', 10, 100),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.5, 1.0),
        subsample=trial.suggest_float('subsample', 0.5, 1.0),
        subsample_freq=1,
        reg_alpha=trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        reg_lambda=trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    )


def objective(trial):
    model = build_lgbm(trial)
    with mlflow.start_run(run_name=f'Trial_{trial.number}_LightGBM_SMOTE_TFIDF_Trigrams'):
        mlflow.set_tags({'experiment_type': 'hyperparameter_tuning', 'batch': BATCH})
        mlflow.log_params({'algo_name': 'LightGBM', **trial.params})
        val_accuracy = accuracy_score(y_val, model.fit(X_fit, y_fit).predict(X_val))
        mlflow.log_metric('val_accuracy', val_accuracy)
    return val_accuracy


study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=N_TRIALS)
print(f'Best validation accuracy: {study.best_value:.4f}')
study.best_params

In [ ]:
best_model = build_lgbm(optuna.trial.FixedTrial(study.best_params))
best_model.fit(X_train, y_train)
y_true = y_test.map(TO_LABEL)
y_pred = pd.Series(best_model.predict(X_test)).map(TO_LABEL)

with mlflow.start_run(run_name='Trial_Best_LightGBM_SMOTE_TFIDF_Trigrams'):
    mlflow.set_tags({'experiment_type': 'hyperparameter_tuning', 'batch': BATCH, 'best': 'true'})
    mlflow.log_params({'algo_name': 'LightGBM', **study.best_params})
    mlflow.log_metric('val_accuracy', study.best_value)
    mlflow.log_metric('accuracy', accuracy_score(y_true, y_pred))
    for label, metrics in classification_report(y_true, y_pred, output_dict=True).items():
        if isinstance(metrics, dict):
            mlflow.log_metrics({f'{label}_{name}': value for name, value in metrics.items()})

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(confusion_matrix(y_true, y_pred, labels=[-1, 0, 1]), annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=[-1, 0, 1], yticklabels=[-1, 0, 1])
    ax.set(xlabel='Predicted', ylabel='Actual', title='Confusion Matrix: best LightGBM')
    mlflow.log_figure(fig, 'confusion_matrix.png')
    plt.show()

    mlflow.sklearn.log_model(best_model, 'LightGBM_model')

print(classification_report(y_true, y_pred))

In [ ]:
optuna.visualization.matplotlib.plot_param_importances(study)
plt.show()

In [ ]:
optuna.visualization.matplotlib.plot_optimization_history(study)
plt.show()

In [ ]:
top_trials = study.trials_dataframe().sort_values('value', ascending=False).head(10)
top_trials.filter(regex='^(number|value|params_)').round(4)